# Fix Categorical-Absence Cache Values

This notebook patches the existing `.mfa_cache` in place. It does not recompute meta-features. For cached rows with `n_cat_features == 0`, it replaces categorical-absence `NaN` values with `0.0` for the categorical-cardinality and categorical-missingness features.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_dir(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pyproject.toml").exists() and (
            candidate / "src" / "mfa"
        ).exists():
            return candidate
    raise RuntimeError("Could not find meta-feature-analysis project directory.")


PROJECT_DIR = find_project_dir()
CACHE_ROOT = PROJECT_DIR / ".mfa_cache"

CAT_ABSENCE_ZERO_COLUMNS = (
    "mean_cat_cardinality",
    "max_cat_cardinality",
    "high_cardinality_fraction",
    "cat_cardinality_to_n_ratio",
    "cat_missing_fraction",
)

print(f"PROJECT_DIR={PROJECT_DIR}")
print(f"CACHE_ROOT={CACHE_ROOT}")

PROJECT_DIR=/work/mherre/tabular-meta-feature-analysis/meta-feature-analysis
CACHE_ROOT=/work/mherre/tabular-meta-feature-analysis/meta-feature-analysis/.mfa_cache


In [2]:
def patch_frame(frame: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    if "n_cat_features" not in frame.columns:
        return frame, {}

    mask = pd.to_numeric(frame["n_cat_features"], errors="coerce").eq(0)
    if not mask.any():
        return frame, {}

    patched = frame.copy()
    changed: dict[str, int] = {}
    for column in CAT_ABSENCE_ZERO_COLUMNS:
        if column not in patched.columns:
            continue
        current = pd.to_numeric(patched.loc[mask, column], errors="coerce")
        change_mask = current.isna() | current.ne(0.0)
        n_changed = int(change_mask.sum())
        if n_changed:
            patched.loc[mask, column] = 0.0
            changed[column] = n_changed
    return patched, changed


def patch_cache_file(path: Path) -> dict[str, object] | None:
    if path.suffix == ".parquet":
        frame = pd.read_parquet(path)
        patched, changed = patch_frame(frame)
        if not changed:
            return None
        patched.to_parquet(path, index=False)
    elif path.suffix == ".pkl":
        obj = pd.read_pickle(path)
        if not isinstance(obj, pd.DataFrame):
            return None
        patched, changed = patch_frame(obj)
        if not changed:
            return None
        patched.to_pickle(path)
    else:
        return None

    return {
        "path": str(path.relative_to(PROJECT_DIR)),
        "rows": int(len(patched)),
        **{f"changed__{column}": count for column, count in changed.items()},
    }


cache_files = sorted(CACHE_ROOT.rglob("*.parquet")) + sorted(CACHE_ROOT.rglob("*.pkl"))
print(f"Scanning {len(cache_files)} parquet/pickle cache files.")

Scanning 875 parquet/pickle cache files.


In [3]:
patch_rows = []
for path in cache_files:
    result = patch_cache_file(path)
    if result is not None:
        patch_rows.append(result)

patch_summary = pd.DataFrame(patch_rows).fillna(0)
print(f"Patched {len(patch_summary)} cache file(s).")
display(patch_summary)

Patched 1 cache file(s).


,path,rows,changed__mean_cat_cardinality,changed__max_cat_cardinality,changed__high_cardinality_fraction,changed__cat_cardinality_to_n_ratio,changed__cat_missing_fraction
0,.mfa_cache/metafeatures/splits/physiochemical_...,1,1,1,1,1,1


In [ ]:
verification_rows = []
for path in cache_files:
    if path.suffix == ".parquet":
        frame = pd.read_parquet(path)
    elif path.suffix == ".pkl":
        obj = pd.read_pickle(path)
        if not isinstance(obj, pd.DataFrame):
            continue
        frame = obj
    else:
        continue

    if "n_cat_features" not in frame.columns:
        continue
    mask = pd.to_numeric(frame["n_cat_features"], errors="coerce").eq(0)
    if not mask.any():
        continue
    row = {"path": str(path.relative_to(PROJECT_DIR)), "zero_cat_rows": int(mask.sum())}
    for column in CAT_ABSENCE_ZERO_COLUMNS:
        if column in frame.columns:
            row[f"remaining_nan__{column}"] = int(frame.loc[mask, column].isna().sum())
    verification_rows.append(row)

verification = pd.DataFrame(verification_rows).fillna(0)
remaining_columns = [
    column for column in verification.columns if column.startswith("remaining_nan__")
]
remaining_total = (
    int(verification[remaining_columns].to_numpy().sum()) if remaining_columns else 0
)
print(f"Remaining categorical-absence NaNs in scanned cache files: {remaining_total}")
display(
    verification.loc[verification[remaining_columns].sum(axis=1).gt(0)]
    if remaining_columns
    else verification
)

Remaining categorical-absence NaNs in scanned cache files: 5


,path,zero_cat_rows,remaining_nan__mean_cat_cardinality,remaining_nan__max_cat_cardinality,remaining_nan__high_cardinality_fraction,remaining_nan__cat_cardinality_to_n_ratio,remaining_nan__cat_missing_fraction
249,.mfa_cache/metafeatures/splits/physiochemical_...,1,1.0,1.0,1.0,1.0,1.0
